# Ask a repository: Qwen2.5-Coder-14B on a Colab GPU

Use a **GPU runtime with at least 40 GB** (A100 40 GB). Put two files at the top of My Drive:
`repo_rag_ask_source.zip` (from the project's `notebooks/` folder) and `repo_rag_index.zip`
(made on the laptop with `repo-rag export-index repo_rag_index.zip`). Then run the cells in order.

The agent is the one evaluated in the Phase 7 bake-offs: the same prompt arm, tools, limits and
retrieval settings, plus a check that every `path:start-end` citation matches a chunk that was
actually retrieved. The model is **provisional**: it passed 4 of 5 pre-registered gates and missed
the invalid-citation gate by one reference. Answers are cited but not verified for correctness.
Without commit history indexed, questions about commits, PRs and issues will have little to use.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!command -v uv >/dev/null || pip install -q uv
# Pinned to the versions in the project's uv.lock. Colab already ships a CUDA build of torch.
!uv pip install --system -q "chromadb==1.5.9" "fastapi==0.162.0" "gitpython==3.2.1" "langchain-chroma==1.1.0" "langchain-core==1.6.13" "langchain-text-splitters==1.1.3" "langgraph==1.2.19" "pydantic==2.14.1" "pydantic-settings==2.16.1" "rank-bm25==0.2.2" "rich==15.0.0" "sentence-transformers==5.7.0" "transformers==4.57.6" "tree-sitter==0.26.0" "tree-sitter-language-pack==1.22.1" "typer==0.27.3" "uvicorn==0.54.0"

In [ ]:
import hashlib, io, os, pathlib, subprocess, sys, zipfile
from google.colab import drive
drive.mount('/content/drive')
SRC = pathlib.Path('/content/repo_rag_src')
SOURCE = pathlib.Path('/content/drive/MyDrive/repo_rag_ask_source.zip')
EXPECTED_SHA256 = '4947d477b80bf394743c5324137ed9994ae0a76c2c32541972166ccd7f30f937'
assert SOURCE.exists(), 'Upload repo_rag_ask_source.zip (from the project notebooks folder) to My Drive'
packed = SOURCE.read_bytes()
assert hashlib.sha256(packed).hexdigest() == EXPECTED_SHA256, 'Source ZIP differs from notebook'
with zipfile.ZipFile(io.BytesIO(packed)) as archive:
    archive.extractall(SRC)
os.environ.update(
    REPO_RAG_DATA_DIR='/content/data', REPO_RAG_COMPUTE_DEVICE='cuda',
    REPO_RAG_EMBEDDING_DTYPE='float32', TOKENIZERS_PARALLELISM='false',
)
sys.path.insert(0, str(SRC))
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN') or ''
except Exception:
    pass  # optional: a Hugging Face token only makes downloads faster
print('unpacked', len(list(SRC.rglob('*.py'))), 'files')

In [ ]:
REPO = 'psf/requests'          # owner/name, as indexed
INDEX_ZIP = pathlib.Path('/content/drive/MyDrive/repo_rag_index.zip')  # from `repo-rag export-index`
BUILD_HISTORY = False          # True also embeds commit history (slow; tools work without it)

Import the index (nothing is re-embedded).

In [ ]:
from repo_rag.config import LOCAL_14B_MODEL, LOCAL_14B_REVISION, Settings
from repo_rag.store.embeddings import get_embeddings
from repo_rag.store.manifest import Manifest
from repo_rag.store.transfer import import_index
from repo_rag.store.vectorstore import VectorStore

assert INDEX_ZIP.exists(), 'Upload repo_rag_index.zip (repo-rag export-index) to My Drive'
# The default answer model is the 27B; this notebook loads the 14B locally.
settings = Settings().model_copy(
    update={'llm_model': LOCAL_14B_MODEL, 'llm_model_revision': LOCAL_14B_REVISION}
)
store = VectorStore(settings, get_embeddings(settings))
print(import_index(INDEX_ZIP, store=store, settings=settings, repos=[REPO]))
entry = Manifest(settings.manifest_path).get(REPO)
assert entry is not None, 'The index ZIP does not contain ' + REPO
print('indexed commit', entry.commit_sha)

Clone the repository at the indexed commit and build the symbol table.

In [ ]:
# Git tools (blame, releases) read a clone pinned to the indexed commit.
CHECKOUT = pathlib.Path('/content/checkout') / REPO.replace('/', '__')
if not CHECKOUT.exists():
    subprocess.run(['git', 'clone', '--quiet', '--filter=blob:none',
                    f'https://github.com/{REPO}.git', str(CHECKOUT)], check=True)
subprocess.run(['git', '-C', str(CHECKOUT), 'checkout', '--quiet', entry.commit_sha], check=True)

from repo_rag.ingest.symbol_indexer import index_symbols
print(index_symbols(REPO, CHECKOUT, settings))   # definitions and call sites, no GPU needed
if BUILD_HISTORY:
    from repo_rag.history.index import index_history
    print(index_history(REPO, CHECKOUT, settings, store, ref=entry.commit_sha))

Load the model once (a few minutes) and build the agent.

In [ ]:
from repo_rag.agent.local_llm import LocalChatModel
from repo_rag.agent.runtime import build_agent

model = LocalChatModel(settings)
print(f'loaded {settings.llm_model} in {model.load_seconds:.0f}s on {model.hardware}')
agent = build_agent(REPO, model, settings, checkout=CHECKOUT)

def ask(question, thread='colab'):
    result = agent.ask(question, thread_id=thread)
    print('ABSTAINED' if result['abstain'] else 'ANSWER', '|', result['failure'] or 'ok',
          '|', result['error'])
    print(result['answer'])
    cites = result['citations']
    print('citations valid:', cites['valid'], '| removed:', cites['removed'],
          '| attached:', cites['attached'], '| uncited:', cites['uncited'])
    print('tools:', [call['name'] for call in result['tool_calls']],
          '| exact refs:', [ref['value'] for ref in result['refs']])
    if result['sources']:
        print('retrieved:', *result['sources'], sep='\n  ')
    return result

from repo_rag.agent.runtime import evaluated_settings
from repo_rag.retrieval.retrievers import HybridRetriever
_retriever = HybridRetriever(VectorStore(evaluated_settings(settings), get_embeddings(settings)),
                             evaluated_settings(settings))

def peek(query, k=10):
    # What search_code would see for this query (the agent shows the model only the top 5).
    for rank, hit in enumerate(_retriever.search(REPO, query, kind='code', k=k, rerank=False), 1):
        meta = hit.document.metadata
        where = f"{meta['path']}:{meta['start_line']}-{meta['end_line']}"
        print(rank, where, meta.get('symbol', ''))

Ask a question.

In [ ]:
QUESTION = 'How are redirects followed, and where is the maximum number of redirects enforced?'
result = ask(QUESTION)
# If an answer abstains, check retrieval vs the model: are the right files in the top 10?
# peek('How does a call to requests.get() reach the network?')

In [ ]:
# Optional: keep asking; an empty line stops. Earlier questions are remembered but not shown
# to the model.
while True:
    question = input('question> ').strip()
    if not question:
        break
    ask(question)

In [ ]:
# Release the GPU and end the session when you are done (saves compute units).
from google.colab import runtime
model.close()
runtime.unassign()